In [7]:
# app.py — Tab 1: Overview
import streamlit as st
import pandas as pd
import numpy as np
from pathlib import Path
import plotly.graph_objects as go

# ---------- CONFIG PATHS ----------
P_PERF = Path("data/analytics/performance_metrics.parquet")     # ret_p, NAV, drawdown, Sharpe_252, VolAnn_20, TE_252
P_RISK = Path("data/analytics/risk_metrics.parquet")            # VaR95_hist, ES95_hist
P_BENCH= Path("data/processed/benchmark_ret.parquet")           # SPY returns (daily)
P_REG  = Path("data/processed/regime_labels.parquet")           # optional: column 'regime'

st.set_page_config(page_title="Portfolio Dashboard", layout="wide")

# ---------- LOADERS ----------
@st.cache_data
def load_perf():
    df = pd.read_parquet(P_PERF)
    df.index = pd.to_datetime(df.index)
    return df

@st.cache_data
def load_risk():
    if P_RISK.exists():
        df = pd.read_parquet(P_RISK)
        df.index = pd.to_datetime(df.index)
        return df
    return pd.DataFrame()

@st.cache_data
def load_bench_nav():
    if not P_BENCH.exists():
        return pd.Series(dtype=float), pd.Series(dtype=float)
    b = pd.read_parquet(P_BENCH)["SPY"]
    b.index = pd.to_datetime(b.index)
    nav_spy = (1 + b.fillna(0)).cumprod()
    return b, nav_spy.rename("NAV_SPY")

@st.cache_data
def load_regimes():
    if not P_REG.exists():
        return pd.Series(dtype=object)
    lab = pd.read_parquet(P_REG).squeeze()
    # chấp nhận cả DataFrame 1 cột hoặc Series
    if isinstance(lab, pd.DataFrame) and lab.shape[1] == 1:
        lab = lab.iloc[:,0]
    lab.index = pd.to_datetime(lab.index)
    lab.name = "regime"
    return lab

# ---------- DATA ----------
perf = load_perf()
risk = load_risk()
ret_spy, nav_spy = load_bench_nav()
regimes = load_regimes()

# ---------- SIDEBAR FILTERS ----------
st.sidebar.header("Filters")
min_d = perf.index.min().date()
max_d = perf.index.max().date()
date_range = st.sidebar.date_input("Date range", value=(min_d, max_d), min_value=min_d, max_value=max_d)

use_regime = st.sidebar.checkbox("Filter by market regime", value=False)
selected_regimes = []
if use_regime and not regimes.empty:
    all_regs = list(regimes.dropna().unique())
    selected_regimes = st.sidebar.multiselect("Select regimes", options=all_regs, default=all_regs)

# ---------- APPLY FILTER ----------
def apply_filter(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df
    df = df.loc[(df.index.date >= date_range[0]) & (df.index.date <= date_range[1])]
    if use_regime and not regimes.empty and selected_regimes:
        reg = regimes.reindex(df.index)
        df = df[reg.isin(selected_regimes)]
    return df

perf_f = apply_filter(perf)
risk_f = apply_filter(risk) if not risk.empty else risk
nav_spy_f = apply_filter(nav_spy.to_frame()).iloc[:,0] if not nav_spy.empty else nav_spy

# ---------- KPIs ----------
st.title("Overview")

col1, col2, col3, col4, col5 = st.columns(5)

# NAV cuối kỳ (portfolio)
nav_last = float(perf_f["NAV"].iloc[-1]) if not perf_f.empty else np.nan
# Max Drawdown
dd_min = float(perf_f["drawdown"].min()) if not perf_f.empty else np.nan
# Mean Sharpe (252)
sh_mean = float(perf_f["Sharpe_252"].dropna().mean()) if "Sharpe_252" in perf_f and not perf_f["Sharpe_252"].dropna().empty else np.nan
# VaR/ES cuối kỳ (nếu có file risk)
var_last = float(risk_f["VaR95_hist"].dropna().iloc[-1]) if not risk_f.empty and "VaR95_hist" in risk_f and not risk_f["VaR95_hist"].dropna().empty else np.nan
es_last  = float(risk_f["ES95_hist"].dropna().iloc[-1])  if not risk_f.empty and "ES95_hist"  in risk_f and not risk_f["ES95_hist"].dropna().empty  else np.nan

col1.metric("NAV (Portfolio)", f"{nav_last:,.3f}" if np.isfinite(nav_last) else "NA")
col2.metric("Max Drawdown", f"{dd_min:.2%}" if np.isfinite(dd_min) else "NA")
col3.metric("Mean Sharpe (252d)", f"{sh_mean:,.3f}" if np.isfinite(sh_mean) else "NA")
col4.metric("VaR95 (last, daily)", f"{var_last:.2%}" if np.isfinite(var_last) else "NA")
col5.metric("ES95 (last, daily)",  f"{es_last:.2%}"  if np.isfinite(es_last)  else "NA")

st.markdown("---")

# ---------- NAV CHART (Portfolio vs SPY) ----------
fig = go.Figure()

# Portfolio NAV
if not perf_f.empty:
    fig.add_trace(go.Scatter(
        x=perf_f.index, y=perf_f["NAV"], mode="lines", name="Portfolio NAV"
    ))

# SPY NAV
if not nav_spy_f.empty:
    fig.add_trace(go.Scatter(
        x=nav_spy_f.index, y=nav_spy_f.values, mode="lines", name="SPY NAV"
    ))

# Regime shading (nếu có)
if not regimes.empty:
    # Dùng nhãn regime theo filter thời gian
    reg_sel = regimes.reindex(perf_f.index if not perf_f.empty else regimes.index).fillna("Unlabeled")
    # tìm các đoạn liên tục theo cùng regime
    if not reg_sel.empty:
        run_start = reg_sel.index[0]
        run_lab = reg_sel.iloc[0]
        for t, lab in zip(reg_sel.index[1:], reg_sel.iloc[1:]):
            if lab != run_lab:
                fig.add_vrect(x0=run_start, x2=t, fillcolor="lightgrey", opacity=0.15, layer="below", line_width=0)
                run_start = t
                run_lab = lab
        # close last segment
        fig.add_vrect(x0=run_start, x2=reg_sel.index[-1], fillcolor="lightgrey", opacity=0.15, layer="below", line_width=0)

fig.update_layout(
    title="NAV: Portfolio vs SPY",
    xaxis_title="Date",
    yaxis_title="NAV (Base = 1.0)",
    legend=dict(orientation="h", y=1.08, x=0),
    margin=dict(l=20, r=20, t=60, b=20),
    height=480
)
st.plotly_chart(fig, use_container_width=True)

# ---------- QUICK STATS TABLE ----------
with st.expander("Summary statistics (current filter)"):
    def safe_stat(s: pd.Series, fn):
        try:
            return fn(s.dropna())
        except Exception:
            return np.nan

    stats = {
        "Start": perf_f.index.min().date() if not perf_f.empty else None,
        "End": perf_f.index.max().date() if not perf_f.empty else None,
        "Obs": len(perf_f),
        "Avg Daily Return": safe_stat(perf_f["ret_p"], np.mean),
        "Std Daily Return": safe_stat(perf_f["ret_p"], np.std),
        "Ann. Vol (≈)": safe_stat(perf_f["ret_p"], lambda x: x.std(ddof=0)*np.sqrt(252)),
        "Ann. Sharpe (≈)": safe_stat(perf_f["ret_p"], lambda x: (x.mean()/x.std(ddof=0))*np.sqrt(252)),
        "Min Drawdown": dd_min,
        "VaR95 (last)": var_last,
        "ES95 (last)": es_last
    }
    df_stats = pd.DataFrame(stats, index=["Value"]).T
    st.dataframe(df_stats)

2025-11-08 16:13:49.371 No runtime found, using MemoryCacheStorageManager
2025-11-08 16:13:49.372 No runtime found, using MemoryCacheStorageManager
2025-11-08 16:13:49.374 No runtime found, using MemoryCacheStorageManager
2025-11-08 16:13:49.375 No runtime found, using MemoryCacheStorageManager
